In [ ]:
!pip install groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.8/143.8 kB 911.4 kB/s eta 0:00:00


In [ ]:
import os
from google.colab import userdata
from groq import Groq
from tqdm.auto import tqdm
import pandas as pd
import json

# Retrieve your key securely from Colab Secrets
os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

# Initialize the client
client = Groq()

# Run a simple completion
chat_completion = client.chat.completions.create(
    messages=[
        {"role": "user", "content": "Explain about LLM in one sentence."}
    ],
    model="qwen/qwen3.8-27b",
)

print(chat_completion.choices[0].message.content)

A Large Language Model (LLM) is an artificial neural network trained on massive amounts of text data to understand, generate, and manipulate human language with high contextual accuracy.


In [ ]:
LLM_MODEL = "openai/gpt-oss-safeguard-20b"

In [ ]:
# Kita mendefinisikan 3 ulasan singkat untuk menguji respon model
teks_uji_1 = "Kameranya buram tapi layarnya cukup terang."
teks_uji_2 = "Baterainya cepat habis, bikin emosi."
teks_uji_3 = "Awalnya ragu karena harganya murah, eh ternyata performanya ngebut sekelas HP flagship."

In [ ]:
# Memberikan instruksi secara langsung tanpa memberikan contoh (Zero-Shot).
prompt_zero_shot = f"""
Ekstrak aspek bisnis dan sentimen dari ulasan berikut.
Ulasan: '{teks_uji_1}'
"""

response_zero = client.chat.completions.create(
    messages=[{"role": "user", "content": prompt_zero_shot}],
    model=LLM_MODEL,
    temperature=0 # Temperature 0 membuat output lebih deterministik/kaku
)

print("--- Hasil Analisis Zero-Shot ---")
print(response_zero.choices[0].message.content)

--- Hasil Analisis Zero-Shot ---
**Aspek Bisnis** | **Sentimen**  
--- | ---  
Kamera | Negatif (buram)  
Layar | Positif (cukup terang)  

**Catatan**: Ulasan menyampaikan opini campuran; kelemahan pada kualitas kamera berbanding positif pada kecerahan layar.


In [ ]:
# Memandu LLM dengan memberikan pola input-output yang diharapkan (Few-Shot).
prompt_few_shot = f"""
Lakukan ekstraksi aspek dan sentimen. Wajib ikuti pola penulisan berikut:

Review: "Suaranya sember dan harganya mahal."
Output: Aspek: Audio (Negatif), Harga (Negatif)

Review: "Desain keren, layar mulus."
Output: Aspek: Desain (Positif), Layar (Positif)

Review: "{teks_uji_2}"
Output:
"""

response_few = client.chat.completions.create(
    messages=[{"role": "user", "content": prompt_few_shot}],
    model=LLM_MODEL,
    temperature=0
)

print("--- Hasil Analisis Few-Shot ---")
print(response_few.choices[0].message.content)

--- Hasil Analisis Few-Shot ---
Output: Aspek: Baterai (Negatif)


In [ ]:
# Memaksa LLM menjabarkan langkah logika (reasoning) sebelum mencetak konklusi (CoT).
prompt_cot = f"""
Analisis ulasan berikut: "{teks_uji_3}"

Sebelum memberikan hasil akhir, wajib jabarkan langkah berpikirmu:
1. Identifikasi entitas fitur produk yang dibahas.
2. Identifikasi kata sifat atau konteks kalimat yang menempel pada entitas tersebut.
3. Tentukan sentimen akhir.
"""

response_cot = client.chat.completions.create(
    messages=[{"role": "user", "content": prompt_cot}],
    model=LLM_MODEL,
    temperature=0
)

print("--- Hasil Analisis Chain-of-Thought (CoT) ---")
print(response_cot.choices[0].message.content)

--- Hasil Analisis Chain-of-Thought (CoT) ---
**Langkah-langkah analisis**  

1. **Identifikasi entitas fitur produk**  
   - *Harga* (disebut “harganya”)  
   - *Performa* (disebut “performanya”)  

2. **Identifikasi kata sifat / konteks yang menempel**  
   - Pada harga: “murah” → indikasi harga rendah.  
   - Pada performa: “ngebut” dan “sekelas HP flagship” → menunjukkan kecepatan dan kualitas tinggi yang biasanya diharapkan pada perangkat kelas atas.  

3. **Tentukan sentimen akhir**  
   - Awal ragu karena harga terjangkau, namun hasilnya performa cepat setara flagship → persepsi berubah menjadi sangat positif.  
   - **Sentimen keseluruhan:** **Positif** (keunggulan performa mengalahkan ekspektasi berdasarkan harga).


In [ ]:
# Pastikan file 'smartphone_reviews_dummy.csv' sudah diunggah ke Colab
try:
    df = pd.read_csv('smartphone_reviews_dummy.csv')
    print(f"Dataset berhasil dimuat dengan total {len(df)} baris ulasan.")
    display(df.head(3))
except FileNotFoundError:
    print("ERROR: File 'smartphone_reviews_dummy.csv' tidak ditemukan. Pastikan file sudah di-upload.")

In [ ]:
# Menyusun instruksi komprehensif (Role + Aturan Main + Few-Shot + JSON Format)
system_prompt = """
Kamu adalah seorang Senior Data Analyst yang bertugas mengekstrak Aspect-Based Sentiment Analysis dari teks ulasan smartphone.

ATURAN KRITIS:
1. Aspek yang diizinkan HANYA: "Layar", "Baterai", "Kamera", "Performa", "Harga", atau "Desain".
2. Sentimen yang diizinkan HANYA: "Positif", "Negatif", atau "Netral".
3. Kamu WAJIB merespons HANYA dengan format objek JSON yang valid.
4. JSON tersebut harus memiliki satu kunci bernama "extracted_mentions" yang berisi List of Dictionaries.

Contoh Output:
{
  "extracted_mentions": [
    {"aspek": "Baterai", "sentimen": "Positif"},
    {"aspek": "Layar", "sentimen": "Negatif"}
  ]
}

Jika teks tidak membahas aspek spesifik apa pun, kembalikan list kosong: {"extracted_mentions": []}
"""

print("System Prompt berhasil dikonfigurasi dan siap dilampirkan ke API.")

In [ ]:
# Menyiapkan list penampung untuk menyimpan data hasil ekstraksi
hasil_absa_mentions = []

print("Memulai proses ekstraksi API untuk seluruh baris data...")

# Iterasi setiap ulasan menggunakan fungsi tqdm untuk menampilkan progress bar
for ulasan in tqdm(df["text"]):

    # JARING PENGAMAN 1: Proteksi terhadap baris data kosong (NaN/Empty String)
    if pd.isna(ulasan) or str(ulasan).strip() == "":
        hasil_absa_mentions.append([]) # Tambahkan list kosong
        continue

    try:
        # Melakukan pemanggilan ke Groq API
        response = client.chat.completions.create(
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": f"Review: {str(ulasan)}"}
            ],
            model=LLM_MODEL,
            temperature=0,

            # JARING PENGAMAN 2: Memaksa API Groove/OpenAI merespons murni format JSON
            response_format={"type": "json_object"}
        )

        # Menangkap balasan dari model
        result_text = response.choices[0].message.content

        # JARING PENGAMAN 3: Parsing string JSON menjadi tipe data Dictionary Python
        result_dict = json.loads(result_text)

        # Mengekstrak daftar (list) sebutan/mentions yang ada di dalam kunci JSON
        mentions = result_dict.get("extracted_mentions", [])

        # Menyimpan hasil list ke dalam penampung utama
        hasil_absa_mentions.append(mentions)

    except Exception as e:
        # Jika terjadi kegagalan (Network Error/Rate Limit), catat error dan lewati
        print(f"\n[ERROR] Gagal memproses ulasan: {e}")
        hasil_absa_mentions.append([])

# Memasukkan hasil ekstraksi utuh kembali ke dalam DataFrame utama
df["raw_mentions_list"] = hasil_absa_mentions

print("\n\nEkstraksi selesai!")

In [ ]:
# Terdapat kemungkinan 1 ulasan memiliki >1 aspek. Kita menggunakan metode explode()
# untuk memecah list tersebut menjadi baris terpisah agar mudah dianalisis secara tabular.

# eratakan (flatten) data berdasarkan list aspek
df_final = df.explode('raw_mentions_list').reset_index(drop=True)

# Mengekstrak kunci dictionary menjadi kolom DataFrame terpisah
# Jika hasil explode adalah dictionary, ambil nilainya. Jika gagal, kembalikan None.
df_final['aspek'] = df_final['raw_mentions_list'].apply(lambda x: x.get('aspek') if isinstance(x, dict) else "None")
df_final['sentimen'] = df_final['raw_mentions_list'].apply(lambda x: x.get('sentimen') if isinstance(x, dict) else "None")

# Membersihkan DataFrame dari kolom sementara
df_final = df_final.drop(columns=['raw_mentions_list'])

# Eksportasi hasil akhir
nama_file_output = 'Final_ABSA_Smartphone.csv'
# Menggunakan delimiter ';' disarankan jika isi ulasan banyak mengandung koma (standar region Indonesia)
df_final.to_csv(nama_file_output, index=False, sep=';')

print(f"Data tabular final berhasil disusun dan disimpan sebagai '{nama_file_output}'.")
print("\nCuplikan Hasil Analisis Akhir:")
display(df_final[['text', 'aspek', 'sentimen']].head(10))